# Ore Grade Decline (OGD) LCA Workflow Notebook

This notebook demonstrates how to use the OGD LCA functions from the `src/` directory.
Each cell is executable and shows the workflow step-by-step.

## Structure
- **src/** - Contains all the reusable functions
- **Functions are imported** at the beginning
- **Each major step** has its own cell
- **Results are displayed** clearly in each cell

## 1. Setup and Imports

Import all necessary libraries and functions from the `src/` directory.

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import bw2data as bd
import bw2calc as bc
from pathlib import Path

# Import all functions from src package
from src import (
    # Activity search functions
    search_activities,
    # find_spanish_electricity,
    get_activity_by_name_and_location,
    get_inventory_flows,
    get_elementary_flow_contributions,
    create_functional_unit,
    
    # OGD calculation functions
    clean_chemical_name,
    chem_comp,
    has_elem,
    cf_calculator,
    create_ogd_method_from_dataframe,
    run_complete_workflow
)

print('✅ All functions imported successfully')

✅ All functions imported successfully


## 2. Brightway25 Setup

Set up the Brightway25 project and check available databases.

In [3]:
# Set up project
project_name = 'CERC'  # Name of the project, choose wisely according to list(bd.projects)
if project_name not in bd.projects:
    bd.projects.create(project_name)
bd.projects.set_current(project_name)

# Check available databases
print(f'Current project: {bd.projects.current}')
print(f'Available databases: {list(bd.databases)}')

# Check if required databases are available
required_dbs = ['ecoinvent-3.4-biosphere', 'ecoinvent-3.4-cutoff']
missing_dbs = [db for db in required_dbs if db not in bd.databases]

if missing_dbs:
    print(f'⚠️  Missing databases: {missing_dbs}')
else:
    print('✅ All required databases available')

Current project: CERC
Available databases: ['ecoinvent-3.4-biosphere', 'ecoinvent-3.4-cutoff']
✅ All required databases available


## 3. Load Ore Grade Decline Data

Load the ore grade decline constants from the Excel file.

In [15]:
# Load OGD data from Excel
ogd_file = Path('Ore-GradeDeclineConstants.xlsx')
if ogd_file.exists():
    OGD_df = pd.read_excel(ogd_file)
    print(f'✅ Loaded {len(OGD_df)} ore grade decline records')
    
    # Remove gold as mentioned in the notebook
    OGD_df = OGD_df[OGD_df['Metal'] != 'Gold']
    print(f'✅ After removing Gold: {len(OGD_df)} records')
    
    # Show first few records
    display(OGD_df.head())
else:
    print(f'❌ File {ogd_file} not found!')

✅ Loaded 18 ore grade decline records
✅ After removing Gold: 17 records


,Metal,Symbol,alpha,beta,URR,CME,k
0,Aluminium,Al,-1.35,0.10,13400000000000000,1040000000000,1041.0
1,Antimony,Sb,-2.06,0.42,66100000000,6790000000,40.0
2,Chromium,Cr,-1.15,0.12,15200000000000,206000000000,18.0
3,Cobalt,Co,-4.86,0.17,2860000000000,2280000000,NaN
4,Copper,Cu,-3.61,0.17,4360000000000,592000000000,170.0


## 4. Search for Activities

Use the generic search function to find activities by name and location.

In [11]:
# Example 1: Generic activity search
print('=== Generic Activity Search ===')

# Search for electricity activities in Spain
spain_electricity_activities = search_activities(
    name='market for electricity, high voltage',
    location='ES',
    limit=1
)

print(f'Found {len(spain_electricity_activities)} electricity activities in Spain:')
for i, act in enumerate(spain_electricity_activities):
    print(f'  {i+1}. {act.get("name")} - {act.get("location")} - {act.get("reference product")}')

=== Generic Activity Search ===
Searching for activities: name='market for electricity, high voltage', location='ES', database='None'
Found 1 matching activities
Found 1 electricity activities in Spain:
  1. market for electricity, high voltage - ES - electricity, high voltage


## 5. Create OGD Method

Create the Ore Grade Decline method using the OGD dataframe.

In [8]:
# Create OGD method from DataFrame
print('=== Creating OGD Method ===')

try:
    method_object, method_data = create_ogd_method_from_dataframe(OGD_df)
    method_name_tuple = method_object.name
    
    print(f'✅ Method created: {method_name_tuple}')
    print(f'   Unit: {method_object.metadata.get("unit")}')
    print(f'   Number of CFs: {method_object.metadata.get("num_cfs")}')
    print(f'   Based on {len(OGD_df)} elements')
    
except Exception as e:
    print(f'❌ Error creating method: {e}')
    # Fallback: show what we have
    if 'OGD_df' in locals():
        print(f'OGD_df has {len(OGD_df)} records')
        display(OGD_df[['Metal', 'Symbol', 'CF1']].head())

=== Creating OGD Method ===
Creating OGD method from DataFrame...
Matching 4078 biosphere flows to 17 elements...
Server hit a 502 for Lead. Retrying...
Server hit a 502 for Pt. Retrying...
Created method data with 0 characterization factors
✅ Successfully overwrote method: ('Cumulative Ore Grade Decline', 'Cumulative ore grade variation', 'Applied to 17 elements')
   - Unit: change in ore grade per kg of metal extracted
   - Number of CFs: 0
   - Based on 17 elements
✅ Method created: ('Cumulative Ore Grade Decline', 'Cumulative ore grade variation', 'Applied to 17 elements')
   Unit: change in ore grade per kg of metal extracted
   Number of CFs: 0
   Based on 17 elements


## 6. Create Functional Unit

Create a functional unit for 268 TWh of Spanish electricity production.

In [ ]:
# Use the first Spanish electricity activity
if spanish_activities:
    electricity_activity = spanish_activities[0]
    print(f'Using activity: {electricity_activity.get("name")}')
    print(f'Reference product: {electricity_activity.get("reference product")}')
    print(f'Unit: {electricity_activity.get("unit")}')
    
    # Create functional unit for 268 TWh
    fu, amount, unit_display = create_functional_unit(
        electricity_activity, 
        amount_twh=268
    )
    
    print(f'✅ Functional unit created:')
    print(f'   Activity: {electricity_activity.get("name")}')
    print(f'   Amount: {amount} {unit_display}')
    print(f'   Functional unit dict: {fu}')
else:
    print('❌ No Spanish electricity activities available')

## 7. Calculate LCIA and Get Elementary Flow Contributions

Calculate the LCIA score and get elementary flow contributions using bw2calc.

In [ ]:
# Calculate elementary flow contributions
if 'fu' in locals() and 'method_name_tuple' in locals():
    print('=== Calculating Elementary Flow Contributions ===')
    
    try:
        results = calculate_lcia_with_bw2calc(fu, method_name_tuple)
        
        if results:
            total_score = results['total_score']
            elementary_contributions = results['elementary_contributions']
            lca = results['lca']
            
            print(f'✅ LCIA calculation completed:')
            print(f'   Total score: {total_score}')
            print(f'   Number of contributions: {len(elementary_contributions)}')
            
            # Show top 10 contributions
            if not elementary_contributions.empty:
                print('   Top 10 elementary flow contributions:')
                display(elementary_contributions.head(10)[['Flow Name', 'Contribution']])
                
                # Show bottom 10 contributions
                print('   Bottom 10 elementary flow contributions:')
                display(elementary_contributions.tail(10)[['Flow Name', 'Contribution']])
            else:
                print('   No contributions found')
        else:
            print('❌ No results returned from LCIA calculation')
            
    except Exception as e:
        print(f'❌ Error calculating LCIA: {e}')
        import traceback
        traceback.print_exc()
else:
    print('❌ Missing functional unit or method. Please run previous cells first.')

## 8. Get Inventory Flows (Optional)

Get the inventory flows for the functional unit using bw2calc.

In [ ]:
# Get inventory flows
if 'fu' in locals():
    print('=== Getting Inventory Flows ===')
    
    try:
        inventory_df = get_inventory_flows(fu)
        
        if not inventory_df.empty:
            print(f'✅ Found {len(inventory_df)} inventory flows')
            print(f'   Total flows: {inventory_df["Amount"].sum()}')
            
            # Show top 10 flows by amount
            print('   Top 10 flows by amount:')
            display(inventory_df.head(10)[['Flow Name', 'Amount', 'Unit']])
            
            # Filter by flow type
            resource_flows = inventory_df[inventory_df['Type'] == 'natural resource']
            print(f'   Natural resource flows: {len(resource_flows)}')
            
            if not resource_flows.empty:
                print('   Top 5 natural resource flows:')
                display(resource_flows.head()[['Flow Name', 'Amount', 'Unit']])
        else:
            print('❌ No inventory flows found')
            
    except Exception as e:
        print(f'❌ Error getting inventory flows: {e}')
else:
    print('❌ Functional unit not available. Please run previous cells first.')

## 9. Complete Workflow (Alternative)

Run the complete workflow with a single function call.

In [ ]:
# Run complete workflow
print('=== Complete Workflow ===')

try:
    # This does everything: loads data, creates method, finds electricity, creates FU, calculates LCIA
    results = run_complete_workflow(OGD_df_path='Ore-GradeDeclineConstants.xlsx', amount_twh=268)
    
    if results:
        print('✅ Complete workflow finished successfully')
        print(f'   Total score: {results["total_score"]}')
        print(f'   Contributions: {len(results["elementary_contributions"])}')
        
        # Show top contributions
        display(results['elementary_contributions'].head(10)[['Flow Name', 'Contribution']])
    else:
        print('❌ Complete workflow returned no results')
        
except Exception as e:
    print(f'❌ Error in complete workflow: {e}')
    import traceback
    traceback.print_exc()

## Summary

This notebook demonstrates the complete OGD LCA workflow using functions from the `src/` directory:

### Key Functions Used:
- `search_activities()` - Generic activity search by name/location
- `find_spanish_electricity()` - Find Spanish high voltage electricity
- `create_ogd_method_from_dataframe()` - Create OGD method
- `create_functional_unit()` - Create functional unit for 268 TWh
- `calculate_lcia_with_bw2calc()` - Calculate LCIA and get contributions
- `get_inventory_flows()` - Get inventory flows
- `run_complete_workflow()` - Run everything automatically

### Workflow Steps:
1. **Setup** - Import functions and set up Brightway25
2. **Load Data** - Load ore grade decline data
3. **Search Activities** - Find activities by name and location
4. **Create Method** - Create OGD method from data
5. **Create Functional Unit** - 268 TWh of Spanish electricity
6. **Calculate LCIA** - Get elementary flow contributions
7. **Get Inventory** - Optional: get inventory flows
8. **Complete Workflow** - Alternative: run everything at once

### Notes:
- All functions use `bw2calc` directly (no CSV files)
- Method name uses **element count** from OGD_df, not CF count
- Results are returned as pandas DataFrames for easy analysis
- Each cell can be run independently to understand the workflow